# 📊 Job Recommendation System — Exploratory Data Analysis

**Mục tiêu notebook này:**
1. Khám phá và hiểu dữ liệu (EDA)
2. Trực quan hóa phân phối dữ liệu
3. Thử nghiệm pipeline TF-IDF + Cosine Similarity
4. Test hệ thống gợi ý việc làm

> 💡 **Lưu ý**: Notebook này import các module từ thư mục `src/`. Toàn bộ business logic được tách biệt và tái sử dụng trong Streamlit app.

## 0. Setup — Import Libraries & Modules

In [ ]:
import os
import sys

# Thêm thư mục gốc project vào sys.path để import `src/`
sys.path.insert(0, os.path.abspath(".."))

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from wordcloud import WordCloud

from src.data_loader import load_data
from src.preprocessing import clean_text
from src.recommender import compute_similarity, recommend_jobs
from src.vectorizer import build_vectorizer

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("viridis")
%matplotlib inline

print("✅ Imports OK")

## 1. Load Dữ Liệu

In [ ]:
# Load dataset từ file CSV
DATA_PATH = '../data/training_data.csv'

df = load_data(DATA_PATH)

print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Kiểm tra thông tin cơ bản
print('=== Thông tin Dataset ===')
print(f'Số lượng records    : {len(df):,}')
print(f'Số cột              : {df.shape[1]}')
print(f'Số job titles độc lập: {df["position_title"].nunique():,}')
print(f'Giá trị null        : {df.isnull().sum().sum()}')
print()
df.info()

## 2. Phân Tích Phân Phối Dữ Liệu (EDA)

In [ ]:
# ── Top 20 Job Titles phổ biến nhất ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Biểu đồ bar: top 20 job titles
top_titles = df['position_title'].value_counts().head(20)
axes[0].barh(top_titles.index[::-1], top_titles.values[::-1], color=plt.cm.viridis_r(np.linspace(0.2, 0.9, 20)))
axes[0].set_title('Top 20 Job Titles Phổ Biến Nhất', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Số lượng')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Histogram: phân phối độ dài mô tả công việc
df_temp = df.copy()
df_temp['desc_len'] = df_temp['job_description'].str.len()
axes[1].hist(df_temp['desc_len'], bins=50, color='#667eea', edgecolor='white', alpha=0.8)
axes[1].set_title('Phân Phối Độ Dài Mô Tả Công Việc', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Số ký tự')
axes[1].set_ylabel('Số lượng')
axes[1].axvline(df_temp['desc_len'].median(), color='red', linestyle='--', label=f'Median: {int(df_temp["desc_len"].median()):,}')
axes[1].legend()

plt.tight_layout()
plt.suptitle('Exploratory Data Analysis — Job Dataset', fontsize=16, fontweight='bold', y=1.02)
plt.savefig('../data/eda_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Biểu đồ đã được lưu tại data/eda_distribution.png')

In [ ]:
# ── Thống kê mô tả về độ dài mô tả công việc ─────────────────────────────────
df['desc_len'] = df['job_description'].str.len()
df['word_count'] = df['job_description'].str.split().str.len()

print('=== Thống kê độ dài Job Description ===')
stats = df[['desc_len', 'word_count']].describe().round(2)
stats.columns = ['Số ký tự', 'Số từ']
stats

## 3. Text Preprocessing

In [ ]:
# Kiểm tra hàm clean_text() với một ví dụ
sample_text = "Apply at https://jobs.example.com! We need Python & ML engineers. Salary: $80,000/year."
cleaned = clean_text(sample_text)

print('=== Ví dụ Text Preprocessing ===')
print(f'Raw text    : {sample_text}')
print(f'Cleaned text: {cleaned}')

In [ ]:
# Áp dụng preprocessing lên toàn bộ DataFrame
# (Giữ notebook gọn: chỉ gọi `clean_text()` từ `src.preprocessing`)
df = df.copy()
df["cleaned_description"] = df["job_description"].apply(clean_text)

print(f"Đã tạo cột cleaned_description: {df['cleaned_description'].notna().sum()} records")
df[["position_title", "job_description", "cleaned_description"]].head(3)

## 4. Word Cloud Visualization

In [ ]:
# ── Word Cloud từ toàn bộ job descriptions đã được làm sạch ──────────────────
all_text = ' '.join(df['cleaned_description'].dropna())

wordcloud = WordCloud(
    width=900, height=450,
    background_color='white',
    colormap='viridis',
    max_words=150,
    min_font_size=10
).generate(all_text)

plt.figure(figsize=(14, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud — Most Common Terms in Job Descriptions', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../data/wordcloud.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Word cloud đã được lưu tại data/wordcloud.png')

## 5. TF-IDF Vectorization

In [ ]:
# Xây dựng TF-IDF vectorizer và transform toàn bộ corpus
vectorizer = build_vectorizer()
tfidf_matrix = vectorizer.fit_transform(df["cleaned_description"])

print(f"\nTF-IDF Matrix shape: {tfidf_matrix.shape}")
print(f"Số lượng features  : {len(vectorizer.vocabulary_)}")
print(
    f"Sparsity           : {1 - (tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])):.2%}"
)

In [ ]:
# ── Hiển thị top 20 từ có TF-IDF score cao nhất trong toàn bộ corpus ─────────
feature_names = vectorizer.get_feature_names_out()
tfidf_scores = tfidf_matrix.sum(axis=0).A1  # Tính tổng TF-IDF theo từng feature

top_features = pd.DataFrame({
    'term': feature_names,
    'tfidf_sum': tfidf_scores
}).sort_values('tfidf_sum', ascending=False).head(20)

plt.figure(figsize=(12, 6))
colors = plt.cm.viridis(np.linspace(0.2, 0.9, 20))
bars = plt.barh(top_features['term'][::-1], top_features['tfidf_sum'][::-1], color=colors)
plt.xlabel('Tổng TF-IDF Score', fontsize=12)
plt.title('Top 20 Terms — Theo Tổng TF-IDF Score', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

top_features.reset_index(drop=True)

## 6. Cosine Similarity — Tính Toán & Phân Tích

In [ ]:
# Tính cosine similarity matrix
cosine_sim = compute_similarity(tfidf_matrix)

print(f'Cosine Sim matrix shape: {cosine_sim.shape}')
print(f'Min similarity: {cosine_sim[cosine_sim > 0].min():.4f}')
print(f'Max similarity (off-diag): {cosine_sim[cosine_sim < 1.0].max():.4f}')
print(f'Mean similarity: {cosine_sim[cosine_sim < 1.0].mean():.4f}')

In [ ]:
# ── Heatmap similarity cho 30 jobs đầu tiên ──────────────────────────────────
N = 30
sim_subset = cosine_sim[:N, :N]

plt.figure(figsize=(14, 12))
sns.heatmap(
    sim_subset,
    cmap='magma',
    xticklabels=False,
    yticklabels=False,
    cbar_kws={'label': 'Cosine Similarity'}
)
plt.title(f'Cosine Similarity Heatmap — {N} Jobs Đầu Tiên', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Test Hệ Thống Gợi Ý

In [ ]:
# ── Test recommend_jobs() với một job title cụ thể ───────────────────────────
TEST_JOB = df['position_title'].iloc[0]  # Lấy job title đầu tiên làm ví dụ

print(f'\n🔍 Tìm công việc tương tự với: "{TEST_JOB}"')
print('=' * 60)

recommendations = recommend_jobs(
    job_title=TEST_JOB,
    df=df,
    cosine_sim=cosine_sim,
    top_n=5
)

if not recommendations.empty:
    for i, row in recommendations.iterrows():
        print(f'\n#{i+1}. {row["position_title"]}')
        print(f'   Similarity: {row["similarity_score"]:.4f} ({row["similarity_score"]*100:.1f}%)')
        print(f'   {row["job_description"][:120]}...')
else:
    print('Không tìm thấy gợi ý.')

In [ ]:
# ── Hiển thị kết quả dạng bảng ───────────────────────────────────────────────
print(f'\n📋 Kết quả gợi ý cho "{TEST_JOB}":')
recommendations[['position_title', 'similarity_score']]

In [ ]:
# (Optional) Nếu bạn muốn gợi ý từ đoạn text/CV, hãy triển khai thêm một hàm riêng
# (ví dụ: vectorize query_text rồi cosine_similarity với tfidf_matrix).
# Notebook này giữ đúng phạm vi baseline: recommend theo job title.

print("ℹ️ Skipped: recommend from free text (not in baseline modules)")

## 8. Phân Tích Distribution Similarity Scores

In [ ]:
# ── Phân phối similarity scores (loại bỏ giá trị đường chéo = 1.0) ───────────
np.fill_diagonal(cosine_sim, 0)  # Đặt đường chéo về 0 để bỏ qua self-similarity
scores_flat = cosine_sim[cosine_sim > 0].flatten()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(scores_flat, bins=80, color='#764ba2', edgecolor='white', alpha=0.8)
axes[0].set_title('Phân Phối Cosine Similarity Scores', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Cosine Similarity')
axes[0].set_ylabel('Tần số')
axes[0].axvline(scores_flat.mean(), color='red', linestyle='--', label=f'Mean: {scores_flat.mean():.3f}')
axes[0].legend()

# Box plot
axes[1].boxplot(scores_flat, patch_artist=True,
                boxprops=dict(facecolor='#667eea', alpha=0.7),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Box Plot — Cosine Similarity', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Cosine Similarity')

plt.tight_layout()
plt.show()

print(f'Mean similarity: {scores_flat.mean():.4f}')
print(f'Median similarity: {np.median(scores_flat):.4f}')
print(f'Std deviation: {scores_flat.std():.4f}')

## 9. Summary & Kết Luận

### ✅ Pipeline hoàn chỉnh:
```
RAW DATA → clean_text() → TF-IDF Vectorization → Cosine Similarity → Top-N Recommendations
```

### 📌 Nhận xét:
- Dataset chứa đa dạng các vị trí công việc với mô tả chi tiết
- TF-IDF + Cosine Similarity là baseline hiệu quả cho text similarity
- Phần lớn các cặp jobs có similarity thấp → hệ thống phân biệt tốt

### 🚀 Hướng cải thiện:
1. **Semantic Search**: BERT / Sentence-Transformers cho context-aware similarity
2. **Hybrid System**: Kết hợp TF-IDF với collaborative filtering
3. **CV Upload**: Cho phép upload file PDF/DOCX để parse và recommend
4. **API**: Xây dựng FastAPI endpoint cho production deployment